In [27]:
import tensorflow as tf
import pandas as pd
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt

In [28]:
train_file_path = "train-data.tsv"
test_file_path = "valid-data.tsv"

In [29]:
from tensorflow.keras.layers import TextVectorization

# Load data into dataframes 
train_df = pd.read_csv("train-data.tsv", sep='\t', names=['label', 'text'])
valid_df = pd.read_csv("valid-data.tsv", sep='\t', names=['label', 'text'])

# Convert labels to numeric: ham=0, spam=1
train_df['label'] = train_df['label'].map({'ham': 0, 'spam': 1})
valid_df['label'] = valid_df['label'].map({'ham': 0, 'spam': 1})

# Define constants for vectorization
MAX_WORDS = 10000
MAX_LEN = 250

# Setup vectorization layer
vectorize_layer = TextVectorization(
    max_tokens=MAX_WORDS,
    output_mode='int',
    output_sequence_length=MAX_LEN
)

# Adapt to training text
vectorize_layer.adapt(train_df['text'].values)

In [30]:
model = tf.keras.Sequential([
    tf.keras.Input(shape=(1,), dtype=tf.string),
    vectorize_layer,
    tf.keras.layers.Embedding(MAX_WORDS, 64, mask_zero=True),
    # Bidirectional allows the model to understand context from both directions
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64)),
    tf.keras.layers.Dense(64, activation='relu'),
    # Dropout prevents overfitting (memorization)
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# Train the model
model.fit(
    train_df['text'], 
    train_df['label'], 
    epochs=10, 
    validation_data=(valid_df['text'], valid_df['label']),
    verbose=1
)

Epoch 1/10
131/131 [==============================] - 38s 154ms/step - loss: 0.2374 - accuracy: 0.9277 - val_loss: 0.0416 - val_accuracy: 0.9878
Epoch 2/10
131/131 [==============================] - 15s 112ms/step - loss: 0.0288 - accuracy: 0.9935 - val_loss: 0.0381 - val_accuracy: 0.9878
Epoch 3/10
131/131 [==============================] - 15s 117ms/step - loss: 0.0098 - accuracy: 0.9981 - val_loss: 0.0353 - val_accuracy: 0.9892
Epoch 4/10
131/131 [==============================] - 15s 112ms/step - loss: 0.0031 - accuracy: 0.9993 - val_loss: 0.0450 - val_accuracy: 0.9856
Epoch 5/10
131/131 [==============================] - 15s 114ms/step - loss: 0.0029 - accuracy: 0.9995 - val_loss: 0.0309 - val_accuracy: 0.9907
Epoch 6/10
131/131 [==============================] - 15s 116ms/step - loss: 9.2778e-04 - accuracy: 1.0000 - val_loss: 0.0360 - val_accuracy: 0.9907
Epoch 7/10
131/131 [==============================] - 15s 113ms/step - loss: 4.8309e-04 - accuracy: 1.0000 - val_loss: 0.0405 

In [31]:
def predict_message(pred_text):
    # Get probability from the model
    probability = model.predict(tf.expand_dims(pred_text, axis=0))[0][0]
    
    # Classify based on 0.5 threshold
    label = "spam" if probability >= 0.5 else "ham"
    
    return [float(probability), label]

pred_text = "how are you doing today?"
prediction = predict_message(pred_text)
print(prediction)

[1.769711889210157e-05, 'ham']


In [32]:
# Replace the code in Cell [19] with this:

test_messages = [
    "how are you doing today",
    "sale today! to stop texts call 98912460324",
    "i dont want to go. can we try it a different day? available sat",
    "our new mobile video service is live. just install on your phone to start watching.",
    "you have won £1000 cash! call to claim your prize.",
    "i'll bring it tomorrow. don't forget the milk.",
    "wow, is your arm alright. that happened to me one time too"
]

test_answers = ["ham", "spam", "ham", "spam", "spam", "ham", "ham"]
passed = True

for msg, ans in zip(test_messages, test_answers):
    prediction = predict_message(msg)
    label = prediction[1]
    
    # Print the result for each message
    print(f"Msg: '{msg[:20]}...' | Expected: {ans} | Predicted: {label} | Prob: {prediction[0]:.4f}")
    
    if label != ans:
        passed = False
        print("^^^ MISMATCH HERE ^^^")

if passed:
    print("\nYou passed the challenge. Great job!")
else:
    print("\nYou haven't passed yet. Keep trying.")

Msg: 'how are you doing to...' | Expected: ham | Predicted: ham | Prob: 0.0000
Msg: 'sale today! to stop ...' | Expected: spam | Predicted: spam | Prob: 0.9924
Msg: 'i dont want to go. c...' | Expected: ham | Predicted: ham | Prob: 0.0000
Msg: 'our new mobile video...' | Expected: spam | Predicted: spam | Prob: 0.9995
Msg: 'you have won £1000 c...' | Expected: spam | Predicted: spam | Prob: 1.0000
Msg: 'i'll bring it tomorr...' | Expected: ham | Predicted: ham | Prob: 0.0000
Msg: 'wow, is your arm alr...' | Expected: ham | Predicted: ham | Prob: 0.0000

You passed the challenge. Great job!
